# IR System — Wuzzuf Job Search Engine (v2)
### Free-text search on job title · Dropdown filters for all structured fields

**Architecture:**
- **TF-IDF search** runs only on the `name` (job title) column
- **Dropdown filters** (hard filters, applied after retrieval):
  - Department / Category
  - Experience Level
  - Employment Type (Full Time / Part Time / Internship / Freelance / Shift Based)
  - Work Mode (On-site / Hybrid / Remote)
  - Governorate / Location
- **Evaluation:** category-based weighted-OR with Precision, Recall, F1, NDCG, MRR, AP


## 1. Imports & Setup

In [ ]:
import re
import math
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet, stopwords
import warnings
warnings.filterwarnings('ignore')
for pkg in ['wordnet','omw-1.4','punkt','averaged_perceptron_tagger_eng','stopwords']:
    nltk.download(pkg, quiet=True)
print("Imports OK")


## 2. Load Data & Parse Structured Columns

In [ ]:
df_raw = pd.read_csv("Cleaned_Wuzzuf_Jobs.csv")
print(f"Raw shape: {df_raw.shape}")
df_raw.head(3)


In [ ]:
# ── Parse structured filter columns from raw data ───────────────────────────
def _parse_employment_type(t: str) -> str:
    """Return the primary employment type from the comma-separated type field."""
    t = t.lower()
    if 'full time'   in t: return 'Full Time'
    if 'internship'  in t: return 'Internship'
    if 'part time'   in t: return 'Part Time'
    if 'freelance'   in t: return 'Freelance'
    if 'shift based' in t: return 'Shift Based'
    return 'Other'
def _parse_work_mode(t: str) -> str:
    t = t.lower()
    if 'remote' in t:                        return 'Remote'
    if 'hybrid' in t:                        return 'Hybrid'
    if re.search(r'on.?site|onsite', t):     return 'On-site'
    return 'Other'
def _parse_department(cat: str) -> str:
    """
    The job category field is pipe-separated:
      Title | Experience Level | Department | tags...
    We want the department (3rd element onward that contains a slash or is long).
    """
    parts = [p.strip() for p in cat.split('|')]
    for p in parts[2:]:
        if len(p) > 3:
            return p.strip()
    return parts[0].strip() if parts else 'Other'
def _parse_experience(cat: str) -> str:
    parts = [p.strip() for p in cat.split('|')]
    if len(parts) > 1:
        lvl = parts[1].strip()
        if lvl in {'Experienced','Entry Level','Manager','Senior Management','Student'}:
            return lvl
    return 'Other'
def _parse_governorate(loc: str) -> str:
    loc = loc.lower()
    if 'cairo'       in loc: return 'Cairo'
    if 'giza'        in loc: return 'Giza'
    if 'alexandria'  in loc: return 'Alexandria'
    if 'sharqia'     in loc: return 'Sharqia'
    if 'monuf'       in loc: return 'Monufya'
    if 'dakahlia'    in loc or 'mansoura' in loc: return 'Dakahlia'
    if 'gharbia'     in loc or 'tanta'    in loc: return 'Gharbia'
    if 'suez'        in loc: return 'Suez'
    if 'ismailia'    in loc: return 'Ismailia'
    if 'south sinai' in loc or 'sharm'    in loc: return 'South Sinai'
    if 'red sea'     in loc or 'hurghada' in loc: return 'Red Sea'
    if 'damietta'    in loc: return 'Damietta'
    if 'beni suef'   in loc: return 'Beni Suef'
    if 'saudi'       in loc: return 'Saudi Arabia'
    if 'dubai'       in loc or 'emirates' in loc or 'uae' in loc: return 'UAE'
    return 'Other'
df = df_raw.copy()
df['employment_type'] = df['type'].apply(_parse_employment_type)
df['work_mode']       = df['type'].apply(_parse_work_mode)
df['department']      = df['job category'].apply(_parse_department)
df['experience']      = df['job category'].apply(_parse_experience)
df['governorate']     = df['location'].apply(_parse_governorate)
print("Structured columns added:")
print()
for col in ['employment_type','work_mode','department','experience','governorate']:
    print(f"  {col}: {df[col].nunique()} unique values")
df[['name','employment_type','work_mode','department','experience','governorate']].head(5)


## 3. Dropdown Option Lists
These are the valid values for each filter. In a UI, these populate dropdowns.

In [ ]:
# Only include values with enough docs to be useful (>=5 jobs)
def _dropdown_options(series: pd.Series, min_count: int = 5) -> list[str]:
    counts = series.value_counts()
    return ['All'] + sorted(counts[counts >= min_count].index.tolist())
DEPT_OPTIONS  = _dropdown_options(df['department'],      min_count=5)
EXP_OPTIONS   = _dropdown_options(df['experience'],      min_count=5)
EMP_OPTIONS   = _dropdown_options(df['employment_type'], min_count=5)
MODE_OPTIONS  = _dropdown_options(df['work_mode'],       min_count=5)
GOV_OPTIONS   = _dropdown_options(df['governorate'],     min_count=5)
print(f"Department options  ({len(DEPT_OPTIONS)-1} values):")
for o in DEPT_OPTIONS[1:]: print(f"  {o}")
print()
print(f"Experience options ({len(EXP_OPTIONS)-1} values):")
for o in EXP_OPTIONS[1:]: print(f"  {o}")
print()
print(f"Employment type options ({len(EMP_OPTIONS)-1} values):")
for o in EMP_OPTIONS[1:]: print(f"  {o}")
print()
print(f"Work mode options ({len(MODE_OPTIONS)-1} values):")
for o in MODE_OPTIONS[1:]: print(f"  {o}")
print()
print(f"Governorate options ({len(GOV_OPTIONS)-1} values):")
for o in GOV_OPTIONS[1:]: print(f"  {o}")


## 4. Text Preprocessing (Job Title Only)

TF-IDF is now built **only on the `name` column** (job title).  
This gives cleaner, more precise relevance scores — the title is the strongest
signal for what a job is about. Structured metadata (location, type, category)
is handled by the hard-filter dropdowns instead.


In [ ]:
lemmatizer = WordNetLemmatizer()
STOP_WORDS  = set(stopwords.words('english'))
def get_wordnet_pos(word: str):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {'J': wordnet.ADJ, 'N': wordnet.NOUN, 'V': wordnet.VERB, 'R': wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)
def preprocess(text: str, keep_tech_symbols: bool = True) -> list[str]:
    text = str(text).lower()
    if keep_tech_symbols:
        text = re.sub(r'[^a-z0-9\s\+\#\.]', ' ', text)
    else:
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) >= 2 and not t.isdigit()]
    tokens = [lemmatizer.lemmatize(t, get_wordnet_pos(t)) for t in tokens]
    return tokens
# ── Tokenize ONLY the job title (name column) ───────────────────────────────
df['tokens'] = df['name'].apply(preprocess)
print("Tokenization done (title-only).")
print()
print("Sample:")
df[['name','tokens']].head(5)


## 5. Inverted Index

In [ ]:
def build_inverted_index(df_input: pd.DataFrame):
    inv_idx    = defaultdict(list)
    doc_store  = {}
    doc_freq   = defaultdict(int)
    N          = len(df_input)
    for doc_id, row in df_input.iterrows():
        tokens             = row['tokens']
        doc_store[doc_id]  = row['name']
        term_counts        = Counter(tokens)
        for term, tf in term_counts.items():
            inv_idx[term].append((doc_id, tf))
        for term in term_counts:
            doc_freq[term] += 1
    return inv_idx, doc_store, doc_freq, N
inv_idx, doc_store, doc_freq, N = build_inverted_index(df)
print(f"Index built — {len(inv_idx):,} unique terms across {N:,} job titles")


## 6. Search Function with Dropdown Filters

`search()` works in two stages:
1. **TF-IDF ranking** on the job title — scores all docs
2. **Hard filters** — applied after ranking; non-matching docs are removed

Any filter set to `'All'` is ignored (all values pass).


In [ ]:
def search(
    query:           str,
    department:      str = 'All',
    experience:      str = 'All',
    employment_type: str = 'All',
    work_mode:       str = 'All',
    governorate:     str = 'All',
    top_k:           int = 20,
    threshold_ratio: float = 0.2,
) -> pd.DataFrame:
    """
    Search job listings by title (TF-IDF) + structured hard filters.
    Parameters
    ----------
    query           : free text search on job title
    department      : filter by department (from DEPT_OPTIONS)
    experience      : filter by experience level (from EXP_OPTIONS)
    employment_type : filter by employment type (from EMP_OPTIONS)
    work_mode       : filter by work mode (from MODE_OPTIONS)
    governorate     : filter by governorate/country (from GOV_OPTIONS)
    top_k           : max results to return
    threshold_ratio : fraction of max TF-IDF score used as minimum cutoff
    Returns
    -------
    pd.DataFrame sorted by relevance score
    """
    # ── Stage 1: TF-IDF scoring on job title ───────────────────────
    scores = defaultdict(float)
    if query.strip():
        query_tokens = preprocess(query)
        for term in query_tokens:
            if term in inv_idx:
                df_term = doc_freq[term]
                idf     = math.log((N + 1) / (df_term + 1)) + 1   # Laplace-smoothed
                for doc_id, tf in inv_idx[term]:
                    scores[doc_id] += tf * idf
        if scores:
            scored_ids = list(scores.keys())
        else:
            scored_ids = []
    else:
        # No text query — start from full dataset, score = 0
        scored_ids = list(df.index)
        scores     = defaultdict(float)
    if not scored_ids:
        return df.iloc[0:0]
    # ── Stage 2: Apply hard filters ────────────────────────────────
    result = df.loc[scored_ids].copy()
    result['score'] = result.index.map(lambda i: scores[i])
    if department      != 'All': result = result[result['department']      == department]
    if experience      != 'All': result = result[result['experience']      == experience]
    if employment_type != 'All': result = result[result['employment_type'] == employment_type]
    if work_mode       != 'All': result = result[result['work_mode']       == work_mode]
    if governorate     != 'All': result = result[result['governorate']     == governorate]
    # ── Stage 3: Sort ──────────────────────────────────────────────
    result = result.sort_values('score', ascending=False).head(top_k)
    display_cols = ['name','company','location','type','job category','score']
    return result[[c for c in display_cols if c in result.columns]]
# ── Smoke tests ────────────────────────────────────────────────────────────
print("Test 1 — Title search only:")
display(search("data analyst", top_k=5))
print("\nTest 2 — Title + filters:")
display(search("software engineer", employment_type="Full Time", work_mode="Remote", top_k=5))
print("\nTest 3 — Filters only (no text query):")
display(search("", department="IT/Software Development", governorate="Cairo", top_k=5))


## 7. Category-Based Evaluation

### Relevance labelling strategy

Because we now have **clean structured columns** parsed from the data,
the relevance labels are computed directly from those columns — not from
keyword classifiers. This removes the main source of noise in the original notebook.

**Facets extracted from a query:**
- Title keywords → matched against `name` column (same as TF-IDF)
- The active **dropdown filters** passed to `search()` define the hard facets

**Tiered relevance:**
- **Tier 2 (strict)**: ALL active filter facets match the document
- **Tier 1 (lenient)**: ≥ 50% of active filter facets match

NDCG uses Tier 2 = 2 points, Tier 1 = 1 point.


In [ ]:
# ?? Metric helpers (Precision / Recall / F1 only) ???????????????????????????
def _precision_recall_f1(retrieved_set, relevant_set):
    tp = len(retrieved_set & relevant_set)
    p  = tp / len(retrieved_set) if retrieved_set else 0.0
    r  = tp / len(relevant_set)  if relevant_set  else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return round(p, 4), round(r, 4), round(f1, 4), tp
print("Metric helpers defined: Precision, Recall, F1 Score")


In [ ]:
def get_relevant_docs(
    department:      str = 'All',
    experience:      str = 'All',
    employment_type: str = 'All',
    work_mode:       str = 'All',
    governorate:     str = 'All',
    min_match_fraction: float = 0.5,
) -> set:
    """
    Build pseudo ground-truth using LENIENT matching only.
    A document is relevant if it matches at least:
        ceil(min_match_fraction * number_of_active_filters)
    Example with 3 active filters:
    - min_match_fraction=0.50 -> ceil(1.5)=2 matches required
    - min_match_fraction=0.67 -> ceil(2.01)=3 matches required
    """
    filters = {
        'department':      department,
        'experience':      experience,
        'employment_type': employment_type,
        'work_mode':       work_mode,
        'governorate':     governorate,
    }
    active = {col: val for col, val in filters.items() if val != 'All'}
    n_active = len(active)
    if n_active == 0:
        return set(df.index.tolist())
    match_score = pd.Series(0, index=df.index)
    for col, val in active.items():
        match_score += (df[col] == val).astype(int)
    min_matches = max(1, math.ceil(min_match_fraction * n_active))
    relevant_docs = set(df[match_score >= min_matches].index.tolist())
    return relevant_docs
# Verify lenient threshold behavior
filters_demo = dict(employment_type='Full Time', work_mode='Remote', governorate='Cairo')
active_n = len(filters_demo)
for frac in [0.5, 0.67, 1.0]:
    rel = get_relevant_docs(**filters_demo, min_match_fraction=frac)
    min_matches = max(1, math.ceil(frac * active_n))
    print(f"min_match_fraction={frac:.2f} -> requires >= {min_matches}/{active_n} filter matches -> {len(rel):,} relevant docs")


## 8. Single-Query Evaluation

In [ ]:
def evaluate_query(
    query:           str   = '',
    department:      str   = 'All',
    experience:      str   = 'All',
    employment_type: str   = 'All',
    work_mode:       str   = 'All',
    governorate:     str   = 'All',
    k:               int   = 20,
    threshold_ratio: float = 0.2,
    min_match_fraction: float = 0.5,
) -> dict:
    """
    Evaluate one search (query + filters) at a given K.
    Metrics used: Precision, Recall, F1 Score (lenient relevance only).
    """
    relevant_docs = get_relevant_docs(
        department=department, experience=experience,
        employment_type=employment_type, work_mode=work_mode,
        governorate=governorate, min_match_fraction=min_match_fraction
    )
    results_df = search(
        query=query, department=department, experience=experience,
        employment_type=employment_type, work_mode=work_mode,
        governorate=governorate, top_k=k, threshold_ratio=threshold_ratio
    )
    retrieved_ids = results_df.index.tolist()
    retrieved_set = set(retrieved_ids)
    p, r, f1, tp = _precision_recall_f1(retrieved_set, relevant_docs)
    return {
        'query': query,
        'department': department,
        'experience': experience,
        'employment_type': employment_type,
        'work_mode': work_mode,
        'governorate': governorate,
        'k': k,
        'retrieved': len(retrieved_set),
        'relevant_pool': len(relevant_docs),
        'true_positives': tp,
        'precision': p,
        'recall': r,
        'f1_score': f1,
        'min_match_fraction': min_match_fraction,
    }
# Quick test
r = evaluate_query(query='data analyst', employment_type='Full Time', governorate='Cairo', k=20)
for k, v in r.items():
    print(f"  {k:<25s}: {v}")


## 9. Batch Evaluation

In [ ]:
def evaluate_batch(test_cases: list[dict], k_values=(10, 20, 50), min_match_fraction: float = 0.5) -> pd.DataFrame:
    """
    Evaluate multiple (query, filters) combinations across K values.
    Metrics used: Precision, Recall, F1 Score.
    """
    rows = []
    for tc in test_cases:
        for k in k_values:
            r = evaluate_query(
                k=k,
                min_match_fraction=min_match_fraction,
                **{kk: tc.get(kk, 'All') if kk != 'query' else tc.get('query', '')
                   for kk in ['query','department','experience','employment_type','work_mode','governorate']}
            )
            rows.append({
                'Query':          r['query'] or '(no text)',
                'Department':     r['department'],
                'Experience':     r['experience'],
                'Emp. Type':      r['employment_type'],
                'Work Mode':      r['work_mode'],
                'Governorate':    r['governorate'],
                'K':              r['k'],
                'Retrieved':      r['retrieved'],
                'Relevant Pool':  r['relevant_pool'],
                'True Positives': r['true_positives'],
                'Precision':      r['precision'],
                'Recall':         r['recall'],
                'F1 Score':       r['f1_score'],
            })
    return pd.DataFrame(rows).sort_values(['Query','K']).reset_index(drop=True)
# ?? Test cases (mix of text-only, filter-only, combined) ????????????????????
TEST_CASES = [
    {'query': 'data analyst'},
    {'query': 'software engineer'},
    {'query': 'graphic designer'},
    {'query': '', 'employment_type': 'Internship'},
    {'query': '', 'work_mode': 'Remote'},
    {'query': '', 'governorate': 'Alexandria'},
    {'query': 'data analyst',        'employment_type': 'Full Time',  'governorate': 'Cairo'},
    {'query': 'python developer',    'work_mode': 'Remote'},
    {'query': 'marketing specialist','employment_type': 'Full Time',  'governorate': 'Giza'},
    {'query': 'hr specialist',       'experience': 'Experienced',     'governorate': 'Cairo'},
    {'query': 'sales engineer',      'employment_type': 'Full Time',  'work_mode': 'On-site'},
    {'query': 'customer service',    'experience': 'Entry Level',     'governorate': 'Cairo'},
]
summary_df = evaluate_batch(TEST_CASES, k_values=(10, 20, 50), min_match_fraction=0.5)
print(f"Evaluation complete ? {len(summary_df)} rows ({len(TEST_CASES)} queries ? 3 K values)")
summary_df


## 10. Summary Statistics

In [ ]:
metric_cols = ['Precision', 'Recall', 'F1 Score']
print("Mean metrics by K:")
display(summary_df.groupby('K')[metric_cols].mean().round(4))
print()
print("Per-query results at K=20:")
k20 = summary_df[summary_df['K'] == 20].copy()
display(k20[['Query','Emp. Type','Work Mode','Governorate','Retrieved','Relevant Pool','True Positives','Precision','Recall','F1 Score']])


## 11. Result Inspector

In [ ]:
def inspect(
    query:           str   = '',
    department:      str   = 'All',
    experience:      str   = 'All',
    employment_type: str   = 'All',
    work_mode:       str   = 'All',
    governorate:     str   = 'All',
    k:               int   = 20,
    min_match_fraction: float = 0.5,
):
    """
    Show retrieved results with lenient relevance labels only.
    Relevant threshold:
      min_matches = ceil(min_match_fraction * number_of_active_filters)
    """
    relevant_docs = get_relevant_docs(
        department=department,
        experience=experience,
        employment_type=employment_type,
        work_mode=work_mode,
        governorate=governorate,
        min_match_fraction=min_match_fraction,
    )
    results = search(
        query=query,
        department=department,
        experience=experience,
        employment_type=employment_type,
        work_mode=work_mode,
        governorate=governorate,
        top_k=k,
    )
    active_filters = {
        kk: vv for kk, vv in {
            'department': department,
            'experience': experience,
            'employment_type': employment_type,
            'work_mode': work_mode,
            'governorate': governorate,
        }.items() if vv != 'All'
    }
    n_active = len(active_filters)
    min_matches = max(1, math.ceil(min_match_fraction * n_active)) if n_active > 0 else 0
    tp = len(set(results.index) & relevant_docs)
    print(f"Query:                  '{query}'")
    print(f"Active filters:          {active_filters}")
    if n_active > 0:
        print(f"Lenient threshold:       >= {min_matches}/{n_active} matching filters")
    else:
        print("Lenient threshold:       no active filters (all docs in pool)")
    print(f"Relevant Pool:           {len(relevant_docs):,} docs")
    print(f"Retrieved:               {len(results)}")
    print(f"True Positives:          {tp}")
    print()
    rows = []
    for doc_id, row in results.iterrows():
        tier = '? Relevant' if doc_id in relevant_docs else '? Not Relevant'
        rows.append({
            'Relevance':  tier,
            'Title':      row['name'],
            'Company':    row['company'],
            'Location':   row['location'],
            'Type':       row['type'],
            'Dept':       df.loc[doc_id, 'department'],
            'Exp':        df.loc[doc_id, 'experience'],
            'Score':      round(row['score'], 3),
        })
    display(pd.DataFrame(rows))
# Example
inspect(query='data analyst', employment_type='Full Time', governorate='Cairo', k=15, min_match_fraction=0.5)


## 12. Interactive Usage (ipywidgets)

Run this cell to get a live search UI with dropdowns inside the notebook.  
Requires: `pip install ipywidgets`


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display as ipy_display, clear_output
    # ── Widgets ────────────────────────────────────────────────────
    w_query = widgets.Text(
        value='', placeholder='e.g. data analyst, python developer',
        description='Job Title:', layout=widgets.Layout(width='500px')
    )
    w_dept = widgets.Dropdown(
        options=DEPT_OPTIONS, value='All', description='Department:'
    )
    w_exp = widgets.Dropdown(
        options=EXP_OPTIONS, value='All', description='Experience:'
    )
    w_emp = widgets.Dropdown(
        options=EMP_OPTIONS, value='All', description='Emp. Type:'
    )
    w_mode = widgets.Dropdown(
        options=MODE_OPTIONS, value='All', description='Work Mode:'
    )
    w_gov = widgets.Dropdown(
        options=GOV_OPTIONS, value='All', description='Governorate:'
    )
    w_k = widgets.IntSlider(
        value=20, min=5, max=1000, step=5, description='Top K:'
    )
    w_btn = widgets.Button(
        description='Search', button_style='primary'
    )
    out = widgets.Output()
    def on_search(_):
        with out:
            clear_output(wait=True)
            results = search(
                query           = w_query.value,
                department      = w_dept.value,
                experience      = w_exp.value,
                employment_type = w_emp.value,
                work_mode       = w_mode.value,
                governorate     = w_gov.value,
                top_k           = w_k.value,
            )
            print(f"{len(results)} results")
            ipy_display(results)
    w_btn.on_click(on_search)
    ui = widgets.VBox([
        widgets.HBox([w_query, w_k, w_btn]),
        widgets.HBox([w_dept, w_exp]),
        widgets.HBox([w_emp, w_mode, w_gov]),
        out
    ])
    ipy_display(ui)
except ImportError:
    print("ipywidgets not installed. Run:  pip install ipywidgets")
    print("Then restart the kernel and re-run this cell.")
    print()
    print("You can still use search() and inspect() directly:")
    print()
    print("  search('data analyst', employment_type='Full Time', governorate='Cairo', top_k=10)")
    print("  inspect('software engineer', work_mode='Remote', k=10)")
